In [1]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import zipfile
from typing import Tuple
from PIL import Image # Використовуємо PIL замість cv2/tifffile

# ─────────────────────────────────────────────────────────
# 1. АРХІТЕКТУРА МОДЕЛІ (SUBIT-TOPOS v2)
# ─────────────────────────────────────────────────────────
class CurvatureField(nn.Module):
    def __init__(self, sigma: float = 1.0):
        super().__init__()
        self.smooth = self._make_gaussian_kernel(sigma)

    def _make_gaussian_kernel(self, sigma: float) -> nn.Conv3d:
        size = int(6 * sigma + 1) | 1
        coords = torch.arange(size).float() - size // 2
        g = torch.exp(-coords**2 / (2 * sigma**2))
        g = g / g.sum()
        kernel = g[:, None, None] * g[None, :, None] * g[None, None, :]
        kernel = kernel.unsqueeze(0).unsqueeze(0)
        conv = nn.Conv3d(1, 1, size, padding=size//2, bias=False)
        conv.weight = nn.Parameter(kernel, requires_grad=False)
        return conv

    def _second_derivative(self, x: torch.Tensor, dim: int) -> torch.Tensor:
        pad = [0, 0, 0, 0, 0, 0]
        pad[-(dim*2+2)] = 1; pad[-(dim*2+1)] = 1
        x_pad = F.pad(x, pad, mode='replicate')
        slices_m = [slice(None)] * x_pad.dim(); slices_c = [slice(None)] * x_pad.dim(); slices_p = [slice(None)] * x_pad.dim()
        d = 2 + dim
        slices_m[d] = slice(0, -2); slices_c[d] = slice(1, -1); slices_p[d] = slice(2, None)
        return x_pad[tuple(slices_m)] - 2 * x_pad[tuple(slices_c)] + x_pad[tuple(slices_p)]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        xs = self.smooth(x)
        hzz = self._second_derivative(xs, 0); hyy = self._second_derivative(xs, 1); hxx = self._second_derivative(xs, 2)
        eigs = torch.stack([hzz.abs(), hyy.abs(), hxx.abs()], dim=1).squeeze(2)
        eigs_sorted, _ = torch.sort(eigs, dim=1)
        l1, l2, l3 = eigs_sorted[:, 0], eigs_sorted[:, 1], eigs_sorted[:, 2]
        eps = 1e-6; norm = l3 + eps
        flat_score = ((1 - l2/norm) * (1 - l1/norm) * (l3 > eps).float()).squeeze(1)
        edge_score = ((l2 / norm) * (1 - l1/norm)).squeeze(1)
        convex_score = ((hzz < 0) & (hyy < 0) & (hxx < 0)).float().squeeze(1)
        concave_score = ((hzz > 0) & (hyy > 0) & (hxx > 0)).float().squeeze(1)
        return torch.stack([flat_score, edge_score, convex_score, concave_score], dim=1)

class SUBITv2Encoder(nn.Module):
    WHEN_SCALES = [1, 2, 4, 8]
    def __init__(self, embed_dim: int = 32):
        super().__init__()
        self.curvature_fields = nn.ModuleList([CurvatureField(sigma=s) for s in self.WHEN_SCALES])
        self.subit_proj = nn.Sequential(
            nn.Conv3d(16, embed_dim // 2, 1), nn.GroupNorm(8, embed_dim // 2), nn.GELU(),
            nn.Conv3d(embed_dim // 2, embed_dim, 1),
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        scale_features = [curv_field(x) for curv_field in self.curvature_fields]
        return self.subit_proj(torch.cat(scale_features, dim=1))

class LearnedEvolutionFtheta(nn.Module):
    def __init__(self, channels: int, n_steps: int = 3):
        super().__init__()
        self.n_steps = n_steps
        self.diffusion_net = nn.Sequential(
            nn.Conv3d(channels, channels, 3, padding=1, groups=channels),
            nn.Conv3d(channels, channels, 1), nn.GroupNorm(8, channels), nn.GELU(),
            nn.Conv3d(channels, 6, 1),
        )
        self.dt = nn.Parameter(torch.ones(1, channels, 1, 1, 1) * 0.1)
        self.proj = nn.Sequential(nn.Conv3d(channels, channels, 3, padding=1), nn.GroupNorm(8, channels), nn.GELU())
        self.stability_gate = nn.Sequential(nn.Conv3d(channels, 1, 1), nn.Sigmoid())

    def anisotropic_step(self, x: torch.Tensor, D_tensor: torch.Tensor) -> torch.Tensor:
        Dxx, Dyy, Dzz = D_tensor[:, 0:1].abs(), D_tensor[:, 1:2].abs(), D_tensor[:, 2:3].abs()
        pad = nn.ReplicationPad3d(1); xp = pad(x)
        d2z = (xp[:, :, 2:, 1:-1, 1:-1] - 2*x + xp[:, :, :-2, 1:-1, 1:-1])
        d2y = (xp[:, :, 1:-1, 2:, 1:-1] - 2*x + xp[:, :, 1:-1, :-2, 1:-1])
        d2x = (xp[:, :, 1:-1, 1:-1, 2:] - 2*x + xp[:, :, 1:-1, 1:-1, :-2])
        return x + self.dt * (Dzz * d2z + Dyy * d2y + Dxx * d2x)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x; D_tensor = self.diffusion_net(x); gate = self.stability_gate(x); h = x
        for _ in range(self.n_steps): h = self.anisotropic_step(h, D_tensor)
        h = self.proj(h)
        return residual + gate * (h - residual)

class ConvBlock3Dv2(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1), nn.GroupNorm(min(8, out_ch), out_ch), nn.GELU(),
            nn.Conv3d(out_ch, out_ch, 3, padding=1), nn.GroupNorm(min(8, out_ch), out_ch), nn.GELU(),
        )
        self.skip = nn.Conv3d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x): return self.block(x) + self.skip(x)

class SUBITAttention3D(nn.Module):
    def __init__(self, feat_ch: int, subit_ch: int):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Conv3d(feat_ch + subit_ch, feat_ch, 1), nn.GroupNorm(min(8, feat_ch), feat_ch), nn.GELU(),
            nn.Conv3d(feat_ch, feat_ch, 1), nn.Sigmoid(),
        )
    def forward(self, feat, subit): return feat * self.gate(torch.cat([feat, subit], dim=1))

class SUBITUNetV2(nn.Module):
    def __init__(self, in_channels=1, features=(32, 64, 128, 256), subit_embed_dim=32):
        super().__init__()
        self.subit_encoder = SUBITv2Encoder(embed_dim=subit_embed_dim)
        self.input_proj = nn.Conv3d(in_channels + subit_embed_dim, features[0], 1)
        self.encoders, self.pools, self.enc_attn = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        ch = features[0]
        for f in features:
            self.encoders.append(ConvBlock3Dv2(ch, f)); self.pools.append(nn.MaxPool3d(2))
            self.enc_attn.append(SUBITAttention3D(f, subit_embed_dim)); ch = f
        self.bottleneck = ConvBlock3Dv2(features[-1], features[-1] * 2)
        self.evolution = LearnedEvolutionFtheta(features[-1] * 2, n_steps=3)
        self.upconvs, self.decoders, self.dec_attn = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        rev_features = list(reversed(features)); ch = features[-1] * 2
        for f in rev_features:
            self.upconvs.append(nn.ConvTranspose3d(ch, f, 2, stride=2))
            self.decoders.append(ConvBlock3Dv2(f * 2, f)); self.dec_attn.append(SUBITAttention3D(f, subit_embed_dim)); ch = f
        self.feature_proj = nn.Conv3d(features[0], features[0], 1)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        subit_emb = self.subit_encoder(x); subit_skips = [subit_emb]
        for _ in range(len(self.encoders) - 1): subit_skips.append(F.avg_pool3d(subit_skips[-1], 2))
        h = self.input_proj(torch.cat([x, subit_emb], dim=1)); skips = []
        for i, (enc, pool, attn) in enumerate(zip(self.encoders, self.pools, self.enc_attn)):
            h = attn(enc(h), subit_skips[i]); skips.append(h); h = pool(h)
        h = self.evolution(self.bottleneck(h))
        for i, (upconv, dec, attn, skip) in enumerate(zip(self.upconvs, self.decoders, self.dec_attn, reversed(skips))):
            h = upconv(h)
            if h.shape != skip.shape: h = F.interpolate(h, size=skip.shape[2:], mode='trilinear', align_corners=False)
            h = attn(dec(torch.cat([h, skip], dim=1)), F.interpolate(subit_emb, size=h.shape[2:], mode='trilinear', align_corners=False))
        return self.feature_proj(h), subit_emb

class AttractorDecoder(nn.Module):
    def __init__(self, channels: int, n_attractor_steps: int = 5):
        super().__init__()
        self.n_steps = n_attractor_steps
        self.ftheta = LearnedEvolutionFtheta(channels, n_steps=1)
        self.omega_head = nn.Sequential(
            nn.Conv3d(channels * 2, channels, 1), nn.GroupNorm(min(8, channels), channels), nn.GELU(),
            nn.Conv3d(channels, 4, 1), nn.Softmax(dim=1)
        )
        self.surface_head = nn.Conv3d(channels, 2, 1)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h = x; trajectory_delta = torch.zeros_like(x)
        for _ in range(self.n_steps):
            h_new = self.ftheta(h)
            trajectory_delta = trajectory_delta + (h_new - h).abs()
            h = h_new
        omega = self.omega_head(torch.cat([h, trajectory_delta], dim=1))
        omega_weight = omega[:, 3:4] * 1.5 + omega[:, 2:3] * 1.2 + omega[:, 1:2] * 0.8 + omega[:, 0:1] * 0.3
        return self.surface_head(h * omega_weight), omega

class OmegaLossV2(nn.Module):
    def __init__(self, alpha=0.4, beta=0.3, gamma=0.2, delta=0.1):
        super().__init__()
        self.alpha, self.beta, self.gamma, self.delta = alpha, beta, gamma, delta

    def forward(self, logits, labels, omega, subit_emb):
        probs = F.softmax(logits, dim=1)
        ce = F.cross_entropy(logits, labels)
        smooth = 1e-5; p, l = probs[:, 1], labels.float()
        inter = (p * l).sum(dim=(1, 2, 3)); union = p.sum(dim=(1, 2, 3)) + l.sum(dim=(1, 2, 3))
        dice = (1 - (2 * inter + smooth) / (union + smooth)).mean()
        s_emb = F.interpolate(subit_emb, size=probs.shape[2:], mode='trilinear', align_corners=False) if subit_emb.shape[2:] != probs.shape[2:] else subit_emb
        curv = (torch.sigmoid(s_emb[:, 0]) * (1 - probs[:, 1])).mean()
        stab = (omega[:, 0] * probs[:, 1]).mean()
        omega_mean = omega.mean(dim=(0, 2, 3, 4))
        omega_reg = -(-(omega_mean * (omega_mean + 1e-8).log()).sum()) * 0.01
        total = self.alpha * ce + (1 - self.alpha) * dice + self.gamma * curv + self.beta * stab + self.delta * omega_reg
        return total, {"ce": ce.item(), "dice": dice.item(), "curvature": curv.item(), "stability": stab.item(), "omega_reg": omega_reg.item(), "total": total.item()}

class VesuviusPipelineV2:
    def __init__(self, device='cuda', lr=1e-4):
        self.device = device
        self.model = SUBITUNetV2().to(device)
        self.attractor = AttractorDecoder(channels=32).to(device)
        self.loss_fn = OmegaLossV2()
        params = list(self.model.parameters()) + list(self.attractor.parameters())
        self.optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-5)

    def train_step_batch(self, ct_batch, mask_batch):
        self.model.train(); self.attractor.train(); self.optimizer.zero_grad()
        x, y = ct_batch.to(self.device), mask_batch.to(self.device)
        features, subit_emb = self.model(x)
        logits, omega = self.attractor(features)
        if logits.shape[2:] != y.shape[1:]:
            logits = F.interpolate(logits, size=y.shape[1:], mode='trilinear', align_corners=False)
            omega = F.interpolate(omega, size=y.shape[1:], mode='trilinear', align_corners=False)
        loss, components = self.loss_fn(logits, y, omega, subit_emb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()
        with torch.no_grad():
            pred = (F.softmax(logits, dim=1)[:, 1] > 0.5).long()
            p, t = pred.float().flatten(), y.float().flatten()
            components["dice_metric"] = ((2*(p*t).sum()+1e-5) / (p.sum()+t.sum()+1e-5)).item()
        return components

# ─────────────────────────────────────────────────────────
# 2. ОБРОБКА ДАНИХ (ТИЛЬКИ PIL)
# ─────────────────────────────────────────────────────────
def read_image(path):
    """Безпечне читання зображень через PIL"""
    try:
        img = Image.open(path)
        return np.array(img)
    except Exception as e:
        print(f"Помилка читання {path}: {e}")
        return None

class VesuviusKaggleDataset(Dataset):
    def __init__(self, ct_dir, mask_dir, chunk_size=(32, 64, 64), stride=(16, 32, 32), augment=True, max_chunks=500):
        self.chunk_size, self.stride, self.augment = chunk_size, stride, augment
        self.ct_files = sorted(glob.glob(os.path.join(ct_dir, "*.*")))
        self.mask_files = sorted(glob.glob(os.path.join(mask_dir, "*.*"))) if mask_dir else []
        
        if len(self.ct_files) == 0: raise ValueError(f"Не знайдено зображень у {ct_dir}")
        img = read_image(self.ct_files[0])
        if len(img.shape) == 3: img = img[:, :, 0]
        self.H, self.W = img.shape
        self.D = len(self.ct_files)
        
        self.chunks = []
        for z in range(0, self.D - chunk_size[0] + 1, stride[0]):
            for y in range(0, self.H - chunk_size[1] + 1, stride[1]):
                for x in range(0, self.W - chunk_size[2] + 1, stride[2]):
                    self.chunks.append((z, y, x))
                    if len(self.chunks) >= max_chunks: break
                if len(self.chunks) >= max_chunks: break
            if len(self.chunks) >= max_chunks: break

    def __len__(self): return len(self.chunks)

    def safe_crop(self, img, y, x, h, w):
        if len(img.shape) == 3: img = img[:, :, 0]
        H, W = img.shape
        patch = np.zeros((h, w), dtype=img.dtype)
        y1, y2 = max(0, y), min(H, y + h)
        x1, x2 = max(0, x), min(W, x + w)
        if y1 < y2 and x1 < x2:
            patch[y1-y:y2-y, x1-x:x2-x] = img[y1:y2, x1:x2]
        return patch

    def __getitem__(self, idx):
        z, y, x = self.chunks[idx]
        d, h, w = self.chunk_size
        ct_slices = [self.safe_crop(read_image(self.ct_files[z+i]), y, x, h, w) for i in range(d)]
        ct_vol = np.stack(ct_slices).astype(np.float32)
        
        if self.mask_files:
            mask_slices = [self.safe_crop(read_image(self.mask_files[z+i]), y, x, h, w) for i in range(d)]
            mask_vol = np.stack(mask_slices).astype(np.int64)
            mask_vol[mask_vol > 0] = 1
        else:
            mask_vol = np.zeros_like(ct_vol, dtype=np.int64)
            
        if self.augment:
            for ax in [0, 1, 2]:
                if np.random.rand() > 0.5:
                    ct_vol = np.flip(ct_vol, axis=ax).copy(); mask_vol = np.flip(mask_vol, axis=ax).copy()
        return torch.from_numpy(ct_vol).unsqueeze(0), torch.from_numpy(mask_vol)

# ─────────────────────────────────────────────────────────
# 3. ТРЕНУВАННЯ ТА ІНФЕРЕНС В ОДНОМУ ПУСКУ
# ─────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 60)
    print("SUBIT-Vesuvius v2 — Full Kaggle Pipeline (PIL Version)")
    print("=" * 60)
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Пристрій: {device}")
    
    base_path = "/kaggle/input/competitions/vesuvius-challenge-surface-detection"
    ct_dir = os.path.join(base_path, "train_images")
    mask_dir = os.path.join(base_path, "train_labels")
    
    # --- КРОК 1: ТРЕНУВАННЯ ---
    print("\n[1/3] Підготовка даних та тренування...")
    dataset = VesuviusKaggleDataset(ct_dir, mask_dir, chunk_size=(32, 64, 64), stride=(32, 128, 128), augment=True, max_chunks=500)
    train_loader = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=2)
    pipeline = VesuviusPipelineV2(device=device)
    
    EPOCHS = 3 # Зменшено до 3 для швидкого проходження пайплайну на Kaggle
    for epoch in range(EPOCHS):
        pbar = tqdm(train_loader, desc=f"Тренування Епоха {epoch+1}/{EPOCHS}")
        for ct_batch, mask_batch in pbar:
            metrics = pipeline.train_step_batch(ct_batch, mask_batch)
            pbar.set_postfix({'Loss': f"{metrics['total']:.4f}", 'Dice': f"{metrics['dice_metric']:.4f}"})
    
    model_path = "/kaggle/working/subit_vesuvius_v2_trained.pt"
    torch.save({'model_state': pipeline.model.state_dict(), 'attractor_state': pipeline.attractor.state_dict()}, model_path)
    print(f"Модель збережено: {model_path}")
    
    # --- КРОК 2: ІНФЕРЕНС ---
    print("\n[2/3] Генерація масок (Sliding Window)...")
    pipeline.model.eval()
    pipeline.attractor.eval()
    
    test_dir = os.path.join(base_path, "test_images")
    test_files = sorted(glob.glob(os.path.join(test_dir, "*.*")))
    out_dir = "/kaggle/working/predictions"
    os.makedirs(out_dir, exist_ok=True)
    
    CHUNK_D, CHUNK_H, CHUNK_W = 32, 128, 128
    results = {}

    with torch.no_grad():
        for f in tqdm(test_files, desc="Інференс"):
            img_id = os.path.basename(f).split('.')[0]
            try:
                img = read_image(f)
            except Exception as e:
                continue
            if img is None: continue
            if len(img.shape) == 3: img = img[:, :, 0]
            
            H, W = img.shape
            final_mask = np.zeros((H, W), dtype=np.uint8)
            pad_H, pad_W = max(0, CHUNK_H - H), max(0, CHUNK_W - W)
            img_padded = np.pad(img, ((0, pad_H), (0, pad_W)), mode='constant')
            
            for y in range(0, img_padded.shape[0], CHUNK_H):
                for x in range(0, img_padded.shape[1], CHUNK_W):
                    crop_2d = img_padded[y:y+CHUNK_H, x:x+CHUNK_W]
                    ct_vol = np.stack([crop_2d]*CHUNK_D).astype(np.float32)
                    x_tensor = torch.from_numpy(ct_vol).unsqueeze(0).unsqueeze(0).to(device)
                    
                    features, _ = pipeline.model(x_tensor)
                    logits, _ = pipeline.attractor(features)
                    mask_3d = (F.softmax(logits, dim=1)[0, 1] > 0.5).cpu().numpy().astype(np.uint8)
                    mask_2d = mask_3d[16]
                    
                    y_end, x_end = min(y + CHUNK_H, H), min(x + CHUNK_W, W)
                    final_mask[y:y_end, x:x_end] = mask_2d[:y_end-y, :x_end-x]
            
            # Збереження через PIL
            mask_img = Image.fromarray(final_mask)
            mask_img.save(os.path.join(out_dir, f"{img_id}.tif"))
            results[img_id] = 1

    # --- КРОК 3: ПАКУВАННЯ В ZIP ---
    print("\n[3/3] Пакування у submission.zip...")
    
    # ВАЖЛИВО: Видаляємо submission.csv, щоб Kaggle не намагався його читати
    csv_path = '/kaggle/working/submission.csv'
    if os.path.exists(csv_path):
        os.remove(csv_path)

    zip_path = "/kaggle/working/submission.zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(out_dir):
            for file in files:
                zipf.write(os.path.join(root, file), file)
                
    print(f"\n✅ Успіх! Файл {zip_path} створено. Можна подавати на конкурс!")

SUBIT-Vesuvius v2 — Full Kaggle Pipeline (PIL Version)
Пристрій: cuda

[1/3] Підготовка даних та тренування...


Тренування Епоха 3/3: 100%|██████████| 108/108 [02:37<00:00,  1.46s/it, Loss=0.0492, Dice=0.9927]


Модель збережено: /kaggle/working/subit_vesuvius_v2_trained.pt

[2/3] Генерація масок (Sliding Window)...


Інференс: 100%|██████████| 1/1 [00:12<00:00, 12.61s/it]


[3/3] Пакування у submission.zip...

✅ Успіх! Файл /kaggle/working/submission.zip створено. Можна подавати на конкурс!
